In [4]:
!pip install -q -U transformers accelerate

In [5]:
from google.colab import drive; drive.mount('/content/drive')
import json, collections, torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForImageTextToText

FP = '/content/drive/MyDrive/CIS 4300/Final Project'
DATA, OUT = f'{FP}/Milestone 1/data', f'{FP}/Milestone 2/chinmay_test'
SPLIT, LIMIT = 'dev', None  # LIMIT = 20 for a quick check

def load(task, split):
    return [json.loads(l) for l in open(f'{DATA}/{task}/{split}.jsonl')]

Mounted at /content/drive


In [6]:
M = 'Qwen/Qwen3.5-4B'
proc = AutoProcessor.from_pretrained(M, min_pixels=256 * 32 * 32, max_pixels=512 * 512)
model = AutoModelForImageTextToText.from_pretrained(M, dtype=torch.bfloat16, device_map='cuda')

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [7]:
def ask(ex, task):
    img = Image.open(f"{DATA}/{ex['image']}").convert('RGB')
    # Celia's prompts.py wording, verbatim
    if task == 'matching':
        q, opts, labels = 'Choose the caption originally written for this cartoon.', ex['choices'], 'A, B, C, D, or E'
    else:
        q, opts, labels = ('Choose the caption that New Yorker caption-contest readers rated as funnier for this cartoon.',
                           [ex['caption_A'], ex['caption_B']], 'A or B')
    letters = 'ABCDE'[:len(opts)]
    text = q + '\n' + '\n'.join(f'{l}: {c}' for l, c in zip(letters, opts)) + f'\nReply with exactly one letter ({labels}).'
    msgs = [{'role': 'user', 'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': text}]}]
    inp = proc.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=True,
                                   return_tensors='pt', enable_thinking=False).to(model.device)
    out = model.generate(**inp, max_new_tokens=16, do_sample=False)
    ans = proc.decode(out[0, inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return ans if len(ans) == 1 and ans in letters else None  # anything else counts as wrong

In [ ]:
def acc(preds, gold):
    return sum(p == g['label'] for p, g in zip(preds, gold)) / len(gold)

for task in ['matching', 'ranking']:
    gold = load(task, SPLIT)[:LIMIT]
    maj = collections.Counter(e['label'] for e in load(task, 'train')).most_common(1)[0][0]
    preds = [ask(e, task) for e in tqdm(gold, desc=task)]
    print(f'{task}: majority ({maj}) {acc([maj] * len(gold), gold):.1%} | qwen {acc(preds, gold):.1%} | invalid {preds.count(None)}')
    if task == 'ranking':
        for t in ['easy', 'hard']:
            i = [k for k, e in enumerate(gold) if e['tier'] == t]
            print(f'  {t}: qwen {acc([preds[k] for k in i], [gold[k] for k in i]):.1%}')
        print(f'  qwen picked B {preds.count("B") / len(preds):.1%} of the time')
    with open(f'{OUT}/qwen_{task}_{SPLIT}.jsonl', 'w') as f:
        for e, p in zip(gold, preds):
            f.write(json.dumps({'id': e.get('instance_id') or e['pair_id'], 'prediction': p, 'label': e['label']}) + '\n')

matching:   0%|          | 0/531 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[t